# 00 · Environment and the first retrieval

## The engineering question
Can we prove that this notebook uses the intended Python environment, local model artifacts, and an actual Qdrant server?

**Learning outcomes:** identify each execution boundary; explain why native model inference and Docker storage are separate; perform a real insert/query; distinguish server readiness from index readiness.

Run `uv sync --locked`, `docker compose up -d`, and `uv run vmc bootstrap` first. Model downloads are explicit setup work. Inference never silently downloads a different model or substitutes example results. The course supports macOS on Apple Silicon; native PyTorch uses CPU or MPS, while Qdrant runs on CPU inside Docker's Linux VM.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from vector_masterclass.lesson import LessonRun
from vector_masterclass.config import root

plt.rcParams.update(
    {"figure.figsize": (8, 4), "axes.spines.top": False, "axes.spines.right": False}
)
lesson = LessonRun("00", "T", docker=True)
print("Kernel:", sys.executable)
print("Workspace:", root())

In [ ]:
import json
from vector_masterclass.preflight import preflight
from vector_masterclass.config import model_config, model_path

environment = preflight()
display(environment)
rows = []
for key in ["dense", "reranker", "sparse", "generator"]:
    manifest = json.loads((model_path(key) / "manifest.json").read_text())
    rows.append(
        {
            "model": key,
            "device": environment["inference_devices"][key],
            "artifact_MiB": manifest["bytes"] / 2**20,
        }
    )
models = lesson.table(pd.DataFrame(rows), "models")
display(models)
fig, ax = plt.subplots()
ax.bar(models.model, models.artifact_MiB, color="#2563eb")
ax.set(
    ylabel="Pinned model files (MiB)", title="Download/storage cost is distinct from runtime memory"
)
lesson.figure(fig)
plt.show()

## A real round trip
We create an isolated collection, embed two sentences, write them, and query the server. `wait=True` waits for the write operation; it does not by itself prove that every segment has an HNSW index. The tiny collection is an API smoke check, not an ANN benchmark.

In [ ]:
from uuid import uuid4
from qdrant_client import QdrantClient, models as qm
from vector_masterclass.config import qdrant_url
from vector_masterclass.models import embed

texts = ["Atlas warranty lasts two years.", "Boreal warranty lasts five years."]
vectors = embed(texts)
client = QdrantClient(url=qdrant_url(), timeout=10)
collection = "vmc_00_" + uuid4().hex[:8]
client.create_collection(
    collection, vectors_config=qm.VectorParams(size=384, distance=qm.Distance.COSINE)
)
try:
    client.upsert(
        collection,
        points=[
            qm.PointStruct(id=i, vector=v.tolist(), payload={"text": texts[i]})
            for i, v in enumerate(vectors)
        ],
        wait=True,
    )
    hits = client.query_points(
        collection, query=embed(["Atlas warranty"], role="query")[0].tolist(), limit=2
    ).points
    display(
        lesson.table(
            pd.DataFrame([{"id": h.id, "score": h.score, "text": h.payload["text"]} for h in hits]),
            "roundtrip",
        )
    )
    first_hit = hits[0].id
finally:
    client.delete_collection(collection)
    client.close()

## Failure, exercise, and production implications
Stop this project's Qdrant service and rerun the readiness cell: it must fail within its timeout. A missing model must point to the bootstrap step instead of reaching the internet. Restart Qdrant before continuing.

**Exercise:** explain why moving the Python process into this Linux container would change access to the Mac's Metal GPU. Add a third sentence and predict the best result before querying.

In production, readiness, persistence, authentication, and index readiness are different checks. This local smoke test establishes none of the deployment's availability guarantees. Use this setup for learning and controlled experiments; use the operations lessons before interpreting it as an operational design. The cleanup removes only this notebook's randomly named collection.

In [ ]:
manifest = lesson.finish(
    {
        "real_server": environment["qdrant"]["version"] == "1.17.1",
        "correct_top_hit": first_hit == 0,
        "embedding_shape": vectors.shape == (2, 384),
        "native_apple_silicon": environment["hardware"]["supported_host"],
    },
    observations={"environment": environment},
)

## References and next steps
Consult the [Qdrant documentation](https://qdrant.tech/documentation/),
the [MiniLM model card](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2),
and [uv's Jupyter guide](https://docs.astral.sh/uv/guides/integration/jupyter/).
The exact pinned configuration is in `configs/models.json`; worked exercises are in `docs/SOLUTIONS.md`.
This notebook's measurements apply to the recorded local run, not a production deployment.